# 💾 Lab 3 — The Transistor Problem (Moore's Law)

Fit a straight line on the log of transistor counts to rediscover Moore's Law.

## 📊 Dataset

A short table of **13 microprocessors** released between 1971 and 2003, with two columns:

| Column | Description |
|---|---|
| years | Year the microprocessor was released |
| **transistors** | Number of transistors on the chip (ranges from ~10³ to ~10⁹) — **target** |

Because the transistor count grows roughly exponentially with time (**Moore's Law**), we model it **on a log scale**.

## 🎯 Task

Find the least-squares straight line:

log₁₀ N ≈ w₀ + w₁ · (t − 1970)

1. Find the coefficients w₀ and w₁ that minimise the MSE.
2. Report the error on the data.
3. Plot the model along with the data points.
4. Predict the transistor count for a chip introduced in 2015.
5. Compare to the IBM Z13 (2015): ~4 × 10⁹ transistors.

ℹ️ With only 13 data points, our goal here is descriptive (fit Moore's Law), not generalisation. In a normal regression task we always split first and (if needed) scale after the split.

**🧭 Workflow:** Load → Transform → Train → Evaluate → Predict → Visualise


## 1️⃣ Import Libraries & Load Data


## 2️⃣ Preprocess the Data

> The target `transistors` spans many orders of magnitude — we take `log10` to linearise it. We also center the year around 1970 so the intercept is interpretable.


the data is skewed

In [ ]:
# Convert transistors count to log scale
log_transistors = np.log10(transistors)

# Center the year data around 1970
years_centered = years - 1970

In [ ]:
log_transistors.shape

## 3️⃣ Train the Model


In [ ]:
# Reshape the data to fit the model
X = years_centered.reshape(-1, 1)
y = log_transistors.reshape(-1, 1)
X.shape

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(X, y)

In [ ]:
# Coefficients
theta_1 = model.intercept_
theta_2 = model.coef_
theta_1, theta_2

## 4️⃣ Evaluate and Predict


In [ ]:
# Predictions
predictions = model.predict(X)
predictions

In [ ]:
# Calculate MS error
mean_squared_error(y, predictions)

In [ ]:
#manual computations
#m = X.shape[0]
#ones = np.ones(m)
#A = np.column_stack((ones, X))
#xhat = la.solve(A.T @ A, A.T @ y)
#xhat

In [ ]:
# Predict the number of transistors in 2015
year_2015 = np.array([[2015 - 1970]])
log_transistors_2015 = model.predict(year_2015)
log_transistors_2015 # this is log_10 of N

In [ ]:
transistors_2015 = 10 ** log_transistors_2015[0]
transistors_2015

## 5️⃣ Visualize the Fitted Model


In [ ]:
# Plot the data points and the fitted model
plt.figure(figsize=(10, 6))
plt.scatter(years, transistors, color='blue', label='Data points')
plt.plot(years, 10 ** predictions, color='red', label='Fitted model')

# Set the scale of the y-axis to logarithmic
plt.yscale('log')

# Label the axes and the plot
plt.xlabel('Year')
plt.ylabel('Number of Transistors (log scale)')
plt.title('Linear Regression Fit to Transistor Count Data')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

## 🎛️ Interactive prediction

Drag the slider to predict the transistor count for any year.


In [ ]:
# 🎛️ Interactive: pick a year and see the model's prediction
try:
    from ipywidgets import interact, IntSlider
    _HAS_WIDGETS = True
except Exception:
    _HAS_WIDGETS = False

import numpy as np
import matplotlib.pyplot as plt

def _predict_year(year=2015):
    logN = model.predict(np.array([[year - 1970]]))[0, 0]
    N = 10 ** logN
    print(f'Predicted transistors in {year}: {N:,.0f}  ({N:.2e})')

    xs = np.arange(1970, max(2025, year + 1) + 1)
    ys = 10 ** model.predict((xs - 1970).reshape(-1, 1)).ravel()

    plt.figure(figsize=(8, 4))
    plt.scatter(years, transistors, color='#1976d2', label='Data', zorder=3)
    plt.plot(xs, ys, color='#ef5350', label='Fitted model')
    plt.scatter([year], [N], color='#ff9800', s=120, edgecolor='black', zorder=4, label=f'Prediction ({year})')
    plt.yscale('log')
    plt.xlabel('Year'); plt.ylabel('Transistors (log scale)')
    plt.title(f"Moore's Law — prediction for year {year}")
    plt.legend(); plt.grid(True, ls='--', alpha=0.4); plt.tight_layout(); plt.show()

if _HAS_WIDGETS:
    interact(_predict_year, year=IntSlider(value=2015, min=1971, max=2030, step=1, description='Year'))
else:
    _predict_year(2015)
